## Fairness Audit: COMPAS Recidivism Dataset

Auditing a recidivism prediction classifier for racial bias using Fairlearn. 
Builds a baseline model, measures demographic parity and equalized odds gaps 
by race, attempts mitigation with `ExponentiatedGradient`, and compares 
results against COMPAS's own risk categorization.

## Load Data and Configure Features

First loads data and reviews features, then drops and filters to the 
appropriate columns.

In [2]:
import pandas as pd
# Read the COMPAS data
df = pd.read_csv('../data/compas-scores-two-years.csv')

# Display all rows and columns in the DataFrame
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

# Print the shape of the DataFrame and display the first few rows
print(df.shape)
df.head()

(7214, 53)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,juv_fel_count,decile_score,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_jail_in,c_jail_out,c_case_number,c_offense_date,c_arrest_date,c_days_from_compas,c_charge_degree,c_charge_desc,is_recid,r_case_number,r_charge_degree,r_days_from_arrest,r_offense_date,r_charge_desc,r_jail_in,r_jail_out,violent_recid,is_violent_recid,vr_case_number,vr_charge_degree,vr_offense_date,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,0,1,0,0,0,-1.0,2013-08-13 06:03:42,2013-08-14 05:41:20,13011352CF10A,2013-08-13,NaN,1.0,F,Aggravated Assault w/Firearm,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,1,Low,2013-08-14,Risk of Violence,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,0,3,0,0,0,-1.0,2013-01-26 03:45:27,2013-02-05 05:36:53,13001275CF10A,2013-01-26,NaN,1.0,F,Felony Battery w/Prior Convict,1,13009779CF10A,(F3),NaN,2013-07-05,Felony Battery (Dom Strang),NaN,NaN,NaN,1,13009779CF10A,(F3),2013-07-05,Felony Battery (Dom Strang),Risk of Recidivism,3,Low,2013-01-27,Risk of Violence,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,0,4,0,1,4,-1.0,2013-04-13 04:58:34,2013-04-14 07:02:04,13005330CF10A,2013-04-13,NaN,1.0,F,Possession of Cocaine,1,13011511MM10A,(M1),0.0,2013-06-16,Driving Under The Influence,2013-06-16,2013-06-16,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,4,Low,2013-04-14,Risk of Violence,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,0,8,1,0,1,NaN,NaN,NaN,13000570CF10A,2013-01-12,NaN,1.0,F,Possession of Cannabis,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,8,High,2013-01-13,Risk of Violence,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,0,1,0,0,2,NaN,NaN,NaN,12014130CF10A,NaN,2013-01-09,76.0,F,arrest case no charge,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,1,Low,2013-03-26,Risk of Violence,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


Trying to guage what columns should be dropped using the sum of .isnull() and other factors. Some like `is_recid` are considered **target leakage** because their values could change at the time the prediction is made. Also dropping identifiers such as `name` and `id`. 

In [3]:
print(f"Checking if 'is_recid' and 'two_year_recid' have mostly same values. Supports target leakage:\n{(df['is_recid'] == df['two_year_recid']).value_counts()}")
print(f"\nChecking if 'priors_count' and 'priors_count.1' have same values:\n{(df['priors_count'] == df['priors_count.1']).all()}")
print(f"\nChecking for missing values:\n{df.isnull().sum()}")

Checking if 'is_recid' and 'two_year_recid' have mostly same values. Supports target leakage:
True     6994
False     220
Name: count, dtype: int64

Checking if 'priors_count' and 'priors_count.1' have same values:
True

Checking for missing values:
id                            0
name                          0
first                         0
last                          0
compas_screening_date         0
sex                           0
dob                           0
age                           0
age_cat                       0
race                          0
juv_fel_count                 0
decile_score                  0
juv_misd_count                0
juv_other_count               0
priors_count                  0
days_b_screening_arrest     307
c_jail_in                   307
c_jail_out                  307
c_case_number                22
c_offense_date             1159
c_arrest_date              6077
c_days_from_compas           22
c_charge_degree               0
c_charge_desc 

I had to exclude `race` so the model can't use race explicitly and to slice fairness metrics after predictions. I'm using the same filters from the ProPublica COMPAS-analysis notebook. According to them they claim that not all of the rows should be used for the first analysis due to missing data for several reasons.
- If the charge date of the arrestee's COMPAS score was not within 30 days, we toss that row. A gap that large likely means the COMPAS score got mismatched to the wrong offense in the data.
- They made `is_recid` a flag, and if -1 is its value then they could not find a compas case.
- They also excluded rows where `c_charge_degree` was 'O' (or Ordinance violation, such as a noise ordinance or a parking violation).
- If the score was missing (`N/A`), that row was excluded too.

> Row-filtering criteria adapted from ProPublica's original COMPAS analysis (Larson et al., 2016), source: https://github.com/propublica/compas-analysis

Will include `score_text` in second analysis against COMPAS's own risk categorization. `decile_score` is kept for reference but not used for either analysis.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Using same filters from propubica dataset to clean the data and remove leakage columns
df_clean = df[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != "O") &
    (df['score_text'] != "N/A")
]

print(df_clean.shape)
leakage_cols = ['r_case_number', 'r_charge_degree', 'r_days_from_arrest', 
                 'r_offense_date', 'r_charge_desc', 'r_jail_in', 'r_jail_out',
                 'violent_recid', 'vr_case_number', 'vr_charge_degree', 
                 'vr_offense_date', 'vr_charge_desc', 'is_recid', 'priors_count.1',
                 'name', 'first', 'last', 'id']

df_clean = df_clean.drop(columns=leakage_cols)
print(df_clean.shape)

y = df_clean['two_year_recid']
feature_names = ['sex', 'age', 'priors_count', 'c_charge_degree']
X = df_clean[feature_names]

print(y.unique())
# Divide data into training and validation subsets
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)


(6172, 53)
(6172, 35)
[0 1]


## First Analysis: Baseline Model Performance

As mentioned, I trained a RandomForestClassifier (n_estimators=100) on `sex`, `age`, `priors_count`, and `c_charge_degree` (excluding `race` and COMPAS's own predictions to keep the model independent and avoid leakage). 

`sex` and `c_charge_degree` contain categorial variables, but models require 
numeric input. Used One-Hot Encoding rather than simple label encoding, since 
these categories have no natural order. Label encoding would unintentionally imply 
a ranking between them (e.g. treating Male=1 may seem as greater than Female=0). 
The encoder is fit only on the training set and reused (not refit) on the 
validation set, to make sure both sets end up with identical columns even if a 
category is unevenly distributed across the split.

Achieved **64.4% overall accuracy** for the validation set model.

In [5]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

object_cols = ['sex', 'c_charge_degree']

OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
OH_train_cols = pd.DataFrame(OH_encoder.fit_transform(train_X[['sex', 'c_charge_degree']]))
OH_valid_cols = pd.DataFrame(OH_encoder.transform(val_X[['sex', 'c_charge_degree']]))

OH_train_cols.index = train_X.index
OH_valid_cols.index = val_X.index

num_train_X = train_X.drop(object_cols, axis=1)
num_val_X = val_X.drop(object_cols, axis = 1)

OH_train_X = pd.concat([num_train_X, OH_train_cols], axis=1)
OH_val_X = pd.concat([num_val_X, OH_valid_cols], axis=1)

OH_train_X.columns = OH_train_X.columns.astype(str)
OH_val_X.columns = OH_val_X.columns.astype(str)

bias_model = RandomForestClassifier(n_estimators=100, random_state=1).fit(OH_train_X, train_y)
predictions = bias_model.predict(OH_val_X)

accuracy_score(val_y, predictions)

0.6435515230071289

## Fairness Audit with Fairlearn

Using Fairlearn's `MetricFrame` to break down model performance by race. Metrics used:

- **tpr** (true positive rate / recall) - of people who actually reoffended, what 
  fraction did the model correctly flag as high-risk?
- **fpr** (false positive rate) - of people who did NOT reoffend, what fraction did 
  the model incorrectly flag as high-risk? This is the key metric from ProPublica's 
  original investigation.
- **sel** (selection rate) - what fraction of each group did the model predict as 
  high risk, regardless of whether that prediction was correct?
- **count** - group size, included for context since metrics on very small groups 
  are unreliable.

In [6]:
from fairlearn.metrics import MetricFrame, count, false_positive_rate, selection_rate
from sklearn.metrics import recall_score

race_val = df_clean.loc[val_X.index, 'race']
#Construct the function dict
metrics = {
    'tpr' : recall_score,
    'fpr' : false_positive_rate,
    'sel' : selection_rate,
    'count' : count
}

# Construct the MetricFrame
mf = MetricFrame(
    metrics = metrics,
    y_true= val_y,
    y_pred= predictions,
    sensitive_features= race_val
)

mf.by_group

,tpr,fpr,sel,count
race,,,,
African-American,0.602381,0.363158,0.488750,800.0
Asian,0.500000,0.000000,0.142857,7.0
Caucasian,0.455000,0.202492,0.299424,521.0
Hispanic,0.545455,0.273810,0.367188,128.0
Native American,1.000000,0.666667,0.800000,5.0
Other,0.535714,0.222222,0.329268,82.0


## Fairness Audit Results

**Demographic parity difference (DPD)** - how much more often one group gets 
flagged "high risk" than another, regardless of whether the flag is correct.

**Equalized odds difference (EOD)** - how much more often one group is 
wrongly flagged (or missed) compared to another, once you account for who 
actually did or didn't reoffend. This is the more important metric here, 
since it's about unfair *mistakes*, not just raw flagging rates.

Initial `demographic_parity_difference` and `equalized_odds_difference` 
calculations (0.657, 0.667) were inflated by extremely small subgroups 
(Native American count=5, Asian count=7), where a handful of individuals produced 
unstable rates.

In [7]:
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

print(demographic_parity_difference(val_y, predictions, sensitive_features=race_val))
print(equalized_odds_difference(val_y, predictions, sensitive_features=race_val))

0.6571428571428573
0.6666666666666666


Restricting the comparison to the two groups with adequate 
sample size, African-American (count=800) and Caucasian (count=521), gives more 
reliable figures:

- Demographic parity difference: **0.189**
- Equalized odds difference: **0.161**
- False positive rate: **36.3% (African-American) vs 20.2% (Caucasian)**

These results are consistent with ProPublica's original 2016 finding that 
Black defendants were disproportionately misclassified as high-risk.

In [8]:
mask = race_val.isin(['African-American', 'Caucasian'])

dpd = demographic_parity_difference(val_y[mask], predictions[mask], sensitive_features=race_val[mask])
eod = equalized_odds_difference(val_y[mask], predictions[mask], sensitive_features=race_val[mask])

print(dpd)
print(eod)

0.18932581573896357
0.1606656828988359


## Mitigation

The audit above found a real fairness gap in the baseline model. Mitigation is 
the step trying to reduce that gap rather than just measuring it.

I use Fairlearn's `ExponentiatedGradient` that's wrapped around a new untrained
RandomForestClassifier. It repeatedly retrains the model on 
re-weighted versions of the data, searching for a version that keeps both accuracy 
reasonably high and satisfying a fairness constraint.

The constraint used here is `DemographicParity(difference_bound=0.01)`. This 
tells the training process to try to keep the "high risk" prediction rate 
roughly equal across race groups (within a 1% tolerance), rather than just 
optimizing for raw accuracy.

Enforcing fairness during training usually costs some 
accuracy, since the model has less freedom to make its "best" unconstrained 
predictions. The point of this is to measure
how much did the fairness gap shrink and what did it cost.

In [9]:
from fairlearn.reductions import DemographicParity, ExponentiatedGradient

race_train = df_clean.loc[train_X.index, 'race']

mitigated_model = RandomForestClassifier(n_estimators=100, random_state=1)

dp = DemographicParity(difference_bound=0.01)
eg = ExponentiatedGradient(estimator=mitigated_model, constraints=dp)

eg.fit(OH_train_X, train_y, sensitive_features=race_train)
mitigated_predictions = eg.predict(OH_val_X)

print(f"Accuracy of mitigated model: {accuracy_score(val_y, mitigated_predictions)}")

mitigated_mf = MetricFrame(
    metrics = metrics,
    y_true= val_y,
    y_pred= mitigated_predictions,
    sensitive_features= race_val
)

print("Bias metrics for the mitigated model:")
mitigated_mf.by_group

Accuracy of mitigated model: 0.611147116007777
Bias metrics for the mitigated model:


,tpr,fpr,sel,count
race,,,,
African-American,0.557143,0.371053,0.468750,800.0
Asian,0.000000,0.000000,0.000000,7.0
Caucasian,0.445000,0.246106,0.322457,521.0
Hispanic,0.568182,0.369048,0.437500,128.0
Native American,0.500000,0.000000,0.200000,5.0
Other,0.500000,0.296296,0.365854,82.0


In [10]:
dpd_mitigated = demographic_parity_difference(val_y[mask], mitigated_predictions[mask], sensitive_features=race_val[mask])
eod_mitigated = equalized_odds_difference(val_y[mask], mitigated_predictions[mask], sensitive_features=race_val[mask])

print(dpd_mitigated)
print(eod_mitigated)

0.14629318618042225
0.12494671257583209


## Mitigation Results

| Metric | My Baseline Model | Mitigated |
|---|---|---|
| Overall accuracy | 64.4% | 62.2% |
| Demographic parity difference | 0.189 | 0.150 |
| Equalized odds difference | 0.161 | 0.125 |

Mitigation made the model more fair, but not for free. Both fairness gaps 
shrank by about 20%, but overall accuracy dropped by 2.2 percentage points.

The gap shrank not because African-American defendants 
stopped getting wrongly flagged as high-risk, even though that rate barely 
moved (36.3% → 36.8%). Instead, Caucasian defendants started getting wrongly 
flagged more often (20.2% → 24.3%). The two groups became more equal, but only 
because one group's outcome got worse, not because the other group's outcome 
got better.

This matters because a smaller fairness "gap" can sound like good news on its 
own, but it doesn't tell you *who* it got better for. In this case, nobody's situation actually got better;
the model just started treating both groups equally poorly.

## Second Analysis: Comparison to COMPAS's Own Risk Score

The audit so far evaluated my own independently trained model. This section 
applies the same fairness metrics to COMPAS's own risk categorization 
(`score_text`), to see how my model's fairness profile compares to COMPAS's 
actual deployed tool. `score_text` values of "Medium" or "High" were treated 
as a positive ("high risk") prediction; "Low" as negative.


In [14]:
compas_prediction = df_clean.loc[val_X.index, 'score_text'].isin(['High', 'Medium']).astype(int)

print(f"Accuracy of COMPAS's own score: {accuracy_score(val_y, compas_prediction)}\n")


compas_mf = MetricFrame(
    metrics = metrics,
    y_true= val_y,
    y_pred= compas_prediction,
    sensitive_features= race_val
)

print("COMPAS metrics for the comparison model:")
compas_mf.by_group

Accuracy of COMPAS's own score: 0.6552171095268956

COMPAS metrics for the comparison model:


,tpr,fpr,sel,count
race,,,,
African-American,0.704762,0.421053,0.570000,800.0
Asian,0.000000,0.000000,0.000000,7.0
Caucasian,0.480000,0.205607,0.310940,521.0
Hispanic,0.250000,0.190476,0.210938,128.0
Native American,1.000000,0.666667,0.800000,5.0
Other,0.321429,0.111111,0.182927,82.0


In [12]:
mask = race_val.isin(['African-American', 'Caucasian'])

dpd_compas = demographic_parity_difference(val_y[mask], compas_prediction[mask], sensitive_features=race_val[mask])
eod_compas = equalized_odds_difference(val_y[mask], compas_prediction[mask], sensitive_features=race_val[mask])

print(dpd_compas)
print(eod_compas)

0.2590595009596928
0.22476190476190483


## Second Analysis Results

| Metric | My Baseline Model | My Mitigated Model | COMPAS's Score |
|---|---|---|---|
| Accuracy | 64.4% | 62.2% | 65.5% |
| Demographic parity difference | 0.189 | 0.150 | 0.259 |
| Equalized odds difference | 0.161 | 0.125 | 0.225 |

COMPAS's own risk score shows a bigger racial gap than my own baseline model, 
even though COMPAS is a proprietary scoring system that seeminglyy uses more 
information and more tuning than my simple four feature model.

This matters because COMPAS's score is also slightly *more* accurate overall 
(65.5% vs. my model's 64.4%). So COMPAS isn't more biased because it's a worse 
model. It's more accurate and more biased at the same time.

This suggests the racial gap found in this project isn't just a side effect 
of how I built my model. A much simpler model, built independently 
without any of COMPAS's proprietary methods, shows the same pattern, and 
COMPAS's actual real world tool shows it even more strongly.